# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [ ]:
!java -version

[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.016s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [ ]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [ ]:
!java DemurrageLegacy.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [ ]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** Нет, это не является дефектом. Метод calc не использует состояние объекта (поля класса), поэтому объявление его static абсолютно оправдано. Это даже хорошая практика: мы явно показываем, что метод является чистой функцией, не зависящей от экземпляра класса. Замечание было бы уместно, если бы класс имел состояние, а метод его игнорировал.

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [ ]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        for (Brigade brigade : brigades) {
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                brigades.remove(brigade);
            }
        }

        String result = "";
        for (int i = 0; i < need; i++) {
            result = result + brigades.get(i).lastName + "; ";
        }

        try {
            FileWriter writer = new FileWriter("наряд.txt");
            writer.write(result);
        } catch (Exception e) {
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanLegacy.java


In [ ]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exception in thread "main" java.util.ConcurrentModificationException
	at java.base/java.util.ArrayList$Itr.checkForComodification(ArrayList.java:1095)
	at java.base/java.util.ArrayList$Itr.next(ArrayList.java:1049)
	at ShiftPlanLegacy.make(ShiftPlanLegacy.java:24)
	at ShiftPlanLegacy.main(ShiftPlanLegacy.java:19)


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [ ]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(Строка="24", Категория="Корректность", Серьёзность="blocker",
         Замечание="Удаление элемента из списка внутри цикла for-each приводит к ConcurrentModificationException, программа аварийно завершается.",
         Предложение="Использовать Iterator.remove() или создать новый список через stream().filter()"),

    dict(Строка="30", Категория="Корректность", Серьёзность="blocker",
         Замечание="Вызов brigades.get(i) без проверки размера списка вызовет IndexOutOfBoundsException, если после фильтрации останется меньше бригад, чем нужно.",
         Предложение="Добавить проверку: если brigades.size() < need, выбросить IllegalArgumentException с осмысленным текстом"),

    dict(Строка="34-37", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="Проглоченное исключение: пустой блок catch скрывает любые ошибки записи в файл (например, нет прав доступа или диск переполнен).",
         Предложение="Логировать исключение (e.printStackTrace() или через логгер) или пробрасывать RuntimeException"),

    dict(Строка="35", Категория="Поведение при ошибках", Серьёзность="major",
         Замечание="Ресурс FileWriter не закрывается. Это приведет к утечке памяти и блокировке файла до сборки мусора.",
         Предложение="Использовать конструкцию try-with-resources для автоматического закрытия файла"),

    dict(Строка="26", Категория="Корректность", Серьёзность="major",
         Замечание="Сравнение вещественного числа brigade.hoursPerShift > 10.0 может работать некорректно из-за ошибок округления double.",
         Предложение="Использовать BigDecimal для точных вычислений или сравнивать с эпсилоном: hours > 10.0 + 1e-9"),

    dict(Строка="22", Категория="Корректность", Серьёзность="major",
         Замечание="Сравнение brigade.admitted == false избыточно и считается плохим тоном.",
         Предложение="Использовать оператор логического НЕ: if (!brigade.admitted)"),

    dict(Строка="9", Категория="Проектирование", Серьёзность="major",
         Замечание="Статическое поле SimpleDateFormat не является потокобезопасным. При одновременном вызове из двух потоков возможны ошибки парсинга дат.",
         Предложение="Использовать локальную переменную внутри метода или java.time.format.DateTimeFormatter (он потокобезопасен)"),

    dict(Строка="29", Категория="Читаемость", Серьёзность="minor",
         Замечание="Конкатенация строк в цикле (result = result + ...) создает множество промежуточных объектов String.",
         Предложение="Использовать StringBuilder для накопления результата"),

    dict(Строка="31", Категория="Проектирование", Серьёзность="minor",
         Замечание="Переменная need не проверяется на отрицательное значение. При need < 0 цикл не выполнится, но логика метода нарушится.",
         Предложение="Добавить проверку в начале метода: if (need < 0) throw new IllegalArgumentException()")
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
ревью

Записано замечаний: 9


,Строка,Категория,Серьёзность,Замечание,Предложение
0,24,Корректность,blocker,Удаление элемента из списка внутри цикла for-e...,Использовать Iterator.remove() или создать нов...
1,30,Корректность,blocker,Вызов brigades.get(i) без проверки размера спи...,Добавить проверку: если brigades.size() < need...
2,34-37,Поведение при ошибках,blocker,Проглоченное исключение: пустой блок catch скр...,Логировать исключение (e.printStackTrace() или...
3,35,Поведение при ошибках,major,Ресурс FileWriter не закрывается. Это приведет...,Использовать конструкцию try-with-resources дл...
4,26,Корректность,major,Сравнение вещественного числа brigade.hoursPer...,Использовать BigDecimal для точных вычислений ...
5,22,Корректность,major,Сравнение brigade.admitted == false избыточно ...,Использовать оператор логического НЕ: if (!bri...
6,9,Проектирование,major,Статическое поле SimpleDateFormat не является ...,Использовать локальную переменную внутри метод...
7,29,Читаемость,minor,Конкатенация строк в цикле (result = result + ...,Использовать StringBuilder для накопления резу...
8,31,Проектирование,minor,Переменная need не проверяется на отрицательно...,Добавить проверку в начале метода: if (need < ...


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [ ]:
%%writefile ShiftPlanFixed.java
import java.io.FileWriter;
import java.io.IOException;
import java.time.LocalDate;
import java.time.format.DateTimeFormatter;
import java.util.ArrayList;
import java.util.Iterator;
import java.util.List;

public class ShiftPlanFixed {

    private static int failedChecks = 0;

    // Вынесли в потокобезопасный класс (замечание 7)
    static DateTimeFormatter dateFormatter = DateTimeFormatter.ofPattern("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        System.out.println("--- Самопроверки ---");

        // 1. Проверка формирования наряда (с сохранением поведения)
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        String result = make(brigades, 2);
        check("Наряд сформирован корректно", "Петров; Козлов; ".equals(result));
        check("Лог обновлен", log.size() == 1);

        // 2. Проверка выброса исключения, если не хватает людей (замечание 2)
        try {
            make(brigades, 5);
            check("Выброс исключения при нехватке бригад", false);
        } catch (IllegalArgumentException e) {
            check("Выброс исключения при нехватке бригад", true);
        }

        // 3. Проверка выброса исключения при отрицательном need (замечание 9)
        try {
            make(brigades, -1);
            check("Выброс исключения при need < 0", false);
        } catch (IllegalArgumentException e) {
            check("Выброс исключения при need < 0", true);
        }

        if (failedChecks > 0) {
            throw new AssertionError("Не пройдено проверок: " + failedChecks);
        }
        System.out.println("\nВсе проверки пройдены.");
    }

    public static String make(List<Brigade> brigades, int need) {
        // Проверка входных данных (замечание 9)
        if (need < 0) {
            throw new IllegalArgumentException("Требуемое количество бригад не может быть отрицательным");
        }

        // Исправление ConcurrentModificationException (замечание 1)
        Iterator<Brigade> iterator = brigades.iterator();
        while (iterator.hasNext()) {
            Brigade brigade = iterator.next();
            // !brigade.admitted вместо == false (замечание 6)
            // Сравнение с эпсилоном для double (замечание 5)
            if (brigade.hoursPerShift > 10.0 + 1e-9 || !brigade.admitted) {
                iterator.remove();
            }
        }

        // Проверка размера перед вызовом get() (замечание 2)
        if (brigades.size() < need) {
            throw new IllegalArgumentException("Недостаточно доступных бригад: запрошено " + need + ", доступно " + brigades.size());
        }

        // Использование StringBuilder (замечание 8)
        StringBuilder result = new StringBuilder();
        for (int i = 0; i < need; i++) {
            result.append(brigades.get(i).lastName).append("; ");
        }

        // try-with-resources для закрытия файла (замечание 4)
        // Обработка исключения вместо пустого catch (замечание 3)
        try (FileWriter writer = new FileWriter("наряд.txt")) {
            writer.write(result.toString());
        } catch (IOException e) {
            System.err.println("Ошибка при записи в файл: " + e.getMessage());
            throw new RuntimeException("Не удалось сохранить наряд", e);
        }

        log.add(LocalDate.now().format(dateFormatter) + " наряд сформирован");
        return result.toString();
    }

    static void check(String name, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + name);
        if (!passed) failedChecks++;
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanFixed.java


In [ ]:
!java ShiftPlanFixed.java

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
--- Самопроверки ---
[OK]   Наряд сформирован корректно
[OK]   Лог обновлен
[OK]   Выброс исключения при нехватке бригад
[OK]   Выброс исключения при need < 0

Все проверки пройдены.


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 0 | blocker | Заменен цикл for-each на Iterator и `iterator.remove()` для избежания ConcurrentModificationException |
| 1 | blocker | Добавлена проверка `if (brigades.size() < need)` с выбросом IllegalArgumentException |
| 2 | blocker | Пустой `catch` заменен на логирование `System.err.println` и проброс RuntimeException |
| 3 | major | Использован `try-with-resources` для автоматического закрытия FileWriter |
| 4 | major | В сравнение double добавлен эпсилон: `hoursPerShift > 10.0 + 1e-9` |
| 5 | major | `admitted == false` заменено на `!brigade.admitted` |
| 6 | major | `SimpleDateFormat` заменен на потокобезопасный `java.time.format.DateTimeFormatter` |
| 7 | minor | Конкатенация строк в цикле заменена на `StringBuilder` |
| 8 | minor | Добавлена проверка `if (need < 0) throw new IllegalArgumentException(...)` |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [ ]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = ""      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = ""         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 0       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 0        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 0         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

Ревью: не указано
Свой запрос: не указан


---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? | 9 замечаний, из них 3 `blocker` и 4 `major`. |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? | Утечку `FileWriter` (замечание 3). Сначала кажется, что раз ошибка ловится в `catch`, то всё в порядке. Но ресурс не закрывается, что приведет к проблемам в продакшене. |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? | Замечание 5 (`admitted == false`). Это скорее вопрос стиля (`minor`), чем реальная ошибка, так как код работает верно. |
| Какое из полученных вами замечаний вы отклонили и на каком основании? | (Напиши здесь то, что тебе написали в PR, либо: "Замечаний не поступало" или "Приняты все"). |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.
   **Ответ:**  Ревью — это ручная проверка логики и архитектуры человеком. Тестирование проверяет поведение (что код делает), а статический анализ ищет шаблонные ошибки (утечки, стиль). Ревью дешевле тестирования на ранних этапах и ловит ошибки проектирования, которые ни тесты, ни анализаторы не видят.
2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?
   **Ответ:** Статический анализатор лучше находит: неиспользуемые переменные, потенциальные NullPointerException, утечки ресурсов, нарушение стиля. Он не найдет: неверно понятое бизнес-требование, неудачную архитектуру, логические ошибки в формулах (если синтаксис верен).
3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.
   **Ответ:** Замечание состоит из: места (где), последствия (почему это плохо) и предложения (как исправить). Плохо: «Перепиши этот цикл». Хорошо: «Строка 24: удаление элемента внутри for-each вызовет ConcurrentModificationException. Предлагаю использовать Iterator.remove()».
4. Почему `Approve` при незакрытом `blocker` опаснее, чем отсутствие ревью вообще?
   **Ответ:** Approve при незакрытом blocker дает ложное чувство безопасности. Автор думает, что код проверен, и сливает его в основную ветку. Отсутствие ревью хотя бы честно сигнализирует о рисках, а ложный Approve маскирует их.
5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?
   **Ответ:** Характеризующий тест фиксирует текущее поведение кода (даже если оно кажется странным) до внесения изменений. Он нужен, чтобы гарантировать, что при рефакторинге или исправлении багов мы случайно не сломали то, что работало правильно.
6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?
   **Ответ:** Крупный PR тяжело читать (усталость, расфокус внимания). Рецензент пропускает мелкие детали, ошибки в условиях и граничные случаи. Мелкие PR легче анализировать, и вероятность найти в них баг выше.

### 7.3 Три вывода по работе

1. Ревью кода — это навык общения. Плохо сформулированное замечание («здесь всё плохо») вызывает защитную реакцию, а хорошее («здесь утечка ресурса, предлагаю try-with-resources») помогает автору улучшить код.
2. Работа с замечаниями требует отделения эмоций от логики. Если рецензент нашел blocker, это не значит, что автор плохой программист, это значит, что код можно сделать надежнее.
3. Инструменты вроде таблиц прослеживаемости и характеризующих тестов дисциплинируют: они заставляют доказывать, что каждое исправление обосновано, и гарантируют, что мы не сломали работающую логику.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [ ]:
#@title Запустить самопроверку и получить отчёт (без части GitHub)
import datetime, os, re, subprocess
import pandas as pd

ФИО_исполнителя = "Минина Дарья Владимировна"  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = True  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя, Группа_исполнителя),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |" % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub (ПРОПУЩЕНО ПО УКАЗАНИЮ ПРЕПОДАВАТЕЛЯ) ---
строки_отчёта += [
    "| Ревью на чужой запрос | ПРОПУЩЕНО (нет пары) |",
    "| Свой запрос | ПРОПУЩЕНО (нет пары) |",
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Code review»

| Параметр | Значение |
|---|---|
| Исполнитель | Минина Дарья Владимировна, группа ГИ-41 |
| Дата отчёта (UTC) | 2026-11-23 09:28 |
| Таблица замечаний | записей 9, заполненных 9, весомых 7 |
| `ShiftPlanFixed.java` | НЕ ПРОЙДЕНО — самопроверок в файле только 4, требуется не менее 5 |
| Ревью на чужой запрос | ПРОПУЩЕНО (нет пары) |
| Свой запрос | ПРОПУЩЕНО (нет пары) |
| Таблица прослеживаемости | заполнена |
| Вопросы и выводы | заполнены |

Скопируйте отчёт выше в описание pull request.



AssertionError: Работа не готова к сдаче: ShiftPlanFixed.java: самопроверок в файле только 4, требуется не менее 5